In [13]:
NUM_AGENTS = 2
HEIGHT = 4
WIDTH = 4
SPAWN_PROB_PER_CELL = 0.05
DESPAWN_PROB_PER_CELL = 0.09

# Parameters
ENV_ITERATIONS = 10000 # 1000000
DISCOUNT = 0.99
EPSILON = 0.1
LEARNING_RATE = 0.00002 # 0.00002
BATCH_SIZE = 32


HIDDEN_FEATURES = 128
HIDDEN_LAYERS = 4
CONV_CHANNELS = [16, 32]
FILENAME = "results_decentralized.csv"

NUM_ENV_CHECKPOINTS = 10


In [14]:
import sys
sys.path.append("../..")
from models.value_cnn import ValueCNNDecentralized
import numpy as np
from orchard.environment import MoveAction, OrchardBasic
from tqdm import tqdm
import torch
from scipy.stats import norm
import copy
import numpy as np
import torch
import matplotlib.pyplot as plt

from models.value_cnn import Transition
from utils import ten_float
from dataclasses import dataclass
from typing import Any
from tadd_helpers.env_functions import *
from tadd_helpers.eval_functions import *
import csv
from pathlib import Path
import ast

if isinstance(CONV_CHANNELS, str):
    CONV_CHANNELS = ast.literal_eval(CONV_CHANNELS)

In [15]:
history = {
    "loss": [],
    "reward_mean": [],
    "reward_std": [],
    "target_mean": [],
    "target_std": [],
    "pred_mean": [],
    "pred_std": [],
    "corr": [],
}

def log_batch_stats(net: ValueCNNDecentralized, batch_size: int):
    """Compute and store one-line summary statistics based on a sample from the replay buffer."""
    with torch.no_grad():
        if len(net.memory) < batch_size:
            return
            
        device = next(net.parameters()).device

        transitions = net.memory.sample(batch_size)
        batch = Transition(*zip(*transitions))

        states = ten_float(np.stack(batch.state), device=device)
        next_states = ten_float(np.stack(batch.new_state), device=device)
        rewards = ten_float(np.array(batch.reward), device=device)

        preds = net.forward(states).squeeze(1)
        
        # CORRECTED: use net.discount instead of net.trainer.discount
        targets = rewards + net.discount * net.target_net(next_states).squeeze(1)

        preds_np, targets_np, rewards_np = preds.cpu().numpy(), targets.cpu().numpy(), rewards.cpu().numpy()
        corr = np.corrcoef(preds_np, targets_np)[0, 1] if len(preds_np) > 1 else np.nan

    history["reward_mean"].append(np.mean(rewards_np))
    history["reward_std"].append(np.std(rewards_np))
    history["target_mean"].append(np.mean(targets_np))
    history["target_std"].append(np.std(targets_np))
    history["pred_mean"].append(np.mean(preds_np))
    history["pred_std"].append(np.std(preds_np))
    history["corr"].append(corr)
def state_to_raw_dict(s: OldState) -> dict:
    return {"apples": s.apples, "agents": s.agents}


apples_in_env = []
checkpoint_states = []

In [16]:
def simulate_step(s: OldState, agent_idx: int, agent_positions: np.ndarray, action_vector: np.ndarray):
    """
    Simulates an agent taking an action. Does not modify in place.
    
    Returns:
        tuple: (rewards: list[int], next_state: State, new_agent_positions: np.ndarray)
            rewards: List of rewards where rewards[i] is the reward obtained by agent i.
    """
    current_agents = s.agents
    current_apples = s.apples
    idx_pos = agent_positions[agent_idx]
    grid_shape = current_agents.shape
    
    new_idx_pos = np.clip(
        idx_pos + action_vector, [0, 0], np.array(grid_shape) - 1
    )
    
    next_agents = current_agents.copy()
    next_apples = current_apples.copy()
    
    next_agents[tuple(new_idx_pos)] += 1
    next_agents[tuple(idx_pos)] -= 1
    
    # The new positions array must be updated
    new_agent_positions = agent_positions.copy()
    new_agent_positions[agent_idx] = new_idx_pos
    
    rewards = [0 for _ in range(len(agent_positions))]
    if next_apples[tuple(new_idx_pos)] > 0:
        rewards[agent_idx] = -1
        next_apples[tuple(new_idx_pos)] -= 1
        
        picker_pos = new_idx_pos
        sum_of_dist_from_agent_to_picker_pos = 0
        for other_agent_idx, other_agent_pos in enumerate(new_agent_positions):
            dist = np.linalg.norm(other_agent_pos - picker_pos)
            sum_of_dist_from_agent_to_picker_pos += dist
        # assign rewards to other agents
        for other_agent_idx, other_agent_pos in enumerate(new_agent_positions):
            if other_agent_idx != agent_idx:
                dist = np.linalg.norm(other_agent_pos - picker_pos).astype(float)
                if sum_of_dist_from_agent_to_picker_pos > 0:
                    rewards[other_agent_idx] = 2 * (dist / sum_of_dist_from_agent_to_picker_pos)
                else:
                    # this should not happen
                    rewards[other_agent_idx] = 0

    return rewards, OldState(apples=next_apples, agents=next_agents), new_agent_positions

### Get CNN Centralized Estimate Value

In [17]:
def Q_team(s_t_plus_1: OldState, r_team: float, V_team: float):
    return r_team + DISCOUNT * V_team

def r_team_func(r_list: list):
    return sum(r_list)

def argmax_a_of_Q_team(s_t, agent_idx: int, agent_positions: np.ndarray, cnn_list: list[ValueCNNDecentralized]):
    max_q = -float('inf')
    best_action = MoveAction.STAY
    for action in MoveAction:
        r_t, s_t_plus_1, new_agent_positions = simulate_step(s_t, agent_idx, agent_positions, action.vector)
        r_team = r_team_func(r_t)
        V_list = []
        for i, cnn in enumerate(cnn_list):
            value = cnn.get_model_reward_prediction_from_raw(state_to_raw_dict(s_t_plus_1), agent_pos=new_agent_positions[i])
            V_list.append(value)
        V_team = sum(V_list)
        q_value = Q_team(s_t_plus_1, r_team, V_team)
        if q_value > max_q:
            max_q = q_value
            best_action = action
    return best_action

In [18]:
V_CNNs: list[ValueCNNDecentralized] = []
for _ in range(NUM_AGENTS):
    V_CNNs.append(ValueCNNDecentralized(HEIGHT, WIDTH, LEARNING_RATE, DISCOUNT, 
                                        HIDDEN_FEATURES, HIDDEN_LAYERS, conv_channels=CONV_CHANNELS))
    
s_0 = old_init_empty_state(HEIGHT, WIDTH)
old_place_agents_randomly(s_0, NUM_AGENTS)
agent_positions = np.argwhere(s_0.agents == 1)

TARGET_UPDATE_FREQUENCY = 100

total_steps = 0

s_t = s_0.copy()
for t in tqdm(range(ENV_ITERATIONS), desc="Training"):
    if t % (ENV_ITERATIONS // NUM_ENV_CHECKPOINTS) == 0 and t > 0:
        apples_in_env.append(np.sum(s_t.apples))
        checkpoint_states.append(s_t.copy())
    cnn = np.random.randint(0, len(agent_positions))
    
    p = np.random.random()
    if p < EPSILON:
        action = MoveAction.get_random_action()
    else:
        action = argmax_a_of_Q_team(s_t, cnn, agent_positions, V_CNNs)
        
    r_D_t_of_c, s_intermediate, new_agent_positions = simulate_step(s_t, cnn, agent_positions, action.vector)
    s_t_plus_1 = s_intermediate.copy()
    old_spawn_apples(s_t_plus_1, SPAWN_PROB_PER_CELL)
    old_despawn_apples(s_t_plus_1, DESPAWN_PROB_PER_CELL)

    for cnn_idx, cnn in enumerate(V_CNNs):
        # c refers to agent c \in C.
        V_CNN_c = cnn
        # 1. Convert states to the network input format
        processed_s_t = V_CNN_c.raw_state_to_nn_input(state_to_raw_dict(s_t), agent_pos=agent_positions[cnn_idx])
        processed_s_t_plus_1 = V_CNN_c.raw_state_to_nn_input(state_to_raw_dict(s_t_plus_1), agent_pos=new_agent_positions[cnn_idx])

        # 2. Add the experience to the replay buffer
        r_idx_D_t_of_c = r_D_t_of_c[cnn_idx]
        V_CNN_c.add_experience(processed_s_t, processed_s_t_plus_1, r_idx_D_t_of_c)

        # 3. Train on a batch ONLY if the buffer is large enough
        if len(V_CNN_c.memory) >= BATCH_SIZE:
            log_batch_stats(V_CNN_c, BATCH_SIZE)
            V_CNN_c.train_batch(BATCH_SIZE)
        
        # 4. Periodically update the target network
        if total_steps % TARGET_UPDATE_FREQUENCY == 0:
            V_CNN_c.update_target_net()

    s_t = s_t_plus_1
    agent_positions = new_agent_positions
    total_steps += 1




Training:   0%|          | 4/10000 [00:00<04:16, 39.02it/s]

Training:   1%|          | 121/10000 [00:16<23:07,  7.12it/s]


KeyboardInterrupt: 

In [ ]:
def plot_training_diagnostics(cnn):
    fig, axs = plt.subplots(3, 2, figsize=(10, 10))
    axs = axs.flatten()

    axs[0].plot(cnn.loss_history)
    axs[0].set_title("Loss")

    axs[1].plot(history["corr"])
    axs[1].set_title("Pred vs Target Correlation")

    axs[2].plot(history["pred_mean"], label="Pred Mean")
    axs[2].plot(history["target_mean"], label="Target Mean")
    axs[2].set_title("Mean Values")
    axs[2].legend()

    axs[3].plot(history["pred_std"], label="Pred Std")
    axs[3].plot(history["target_std"], label="Target Std")
    axs[3].set_title("Std Dev Values")
    axs[3].legend()

    axs[4].plot(history["reward_mean"], label="Reward Mean")
    axs[4].plot(history["reward_std"], label="Reward Std")
    axs[4].set_title("Reward Stats")
    axs[4].legend()

    axs[5].axis("off")
    plt.tight_layout()
    plt.show()

plot_training_diagnostics(V_CNN_c)


In [ ]:
def optimal_learned_policy(s: State, agent_idx: int, agent_positions: np.ndarray):
    return argmax_a_of_Q_team(s, agent_idx, agent_positions, V_CNNs)

In [ ]:
def evaluate_policy(policy_func, num_steps: int = 1000):
    trace_data: list[dict] = []
    total_rewards: list[int] = []
    s_0 = old_init_empty_state(HEIGHT, WIDTH)
    old_place_agents_randomly(s_0, NUM_AGENTS)
    agent_positions = np.argwhere(s_0.agents == 1)
    s_t = s_0.copy()
    for t in range(num_steps):  # fixed episode length
        c = np.random.randint(0, len(agent_positions))
        action = policy_func(s_t, c, agent_positions)
        r_t, s_intermediate, agent_positions = simulate_step(s_t, c, agent_positions, action.vector)
        r_team = r_team_func(r_t)
        s_t_plus_1 = s_intermediate.copy()
        old_spawn_apples(s_t_plus_1, SPAWN_PROB_PER_CELL)
        old_despawn_apples(s_t_plus_1, DESPAWN_PROB_PER_CELL)
        total_rewards.append(r_team)
        trace_data.append({
            'step': t,
            'initial_state': s_t.copy(),
            'agent_idx_acting': c,
            'action_taken': action,
            'individual_rewards': r_t.copy(),
            'team_reward': r_team,
            'final_state_of_step': s_t_plus_1.copy() # State after all environment updates
        })
        s_t = s_t_plus_1
    return total_rewards, trace_data

In [ ]:
random_rewards, random_data = evaluate_policy(old_random_policy, num_steps=1000)
nearest_rewards, nearest_data = evaluate_policy(old_nearest_policy, num_steps=1000)
learned_rewards, learned_data = evaluate_policy(optimal_learned_policy, num_steps=1000)
print(f"Random Policy: Mean Reward = {np.mean(random_rewards):.4f}, Std Dev = {np.std(random_rewards):.4f}")
print(f"Nearest Policy: Mean Reward = {np.mean(nearest_rewards):.4f}, Std Dev = {np.std(nearest_rewards):.4f}")
print(f"Learned Policy: Mean Reward = {np.mean(learned_rewards):.4f}, Std Dev = {np.std(learned_rewards):.4f}")


In [ ]:
def print_evaluation_trace(trace_data, num_steps_to_print: int = 100):
    print("\n--- Evaluation Trace (First {} Steps) ---".format(num_steps_to_print))
    for i, step_data in enumerate(trace_data):
        if i >= num_steps_to_print:
            break
        print(f"\n--- Step {step_data['step']} ---")
        print(f"Agent {step_data['agent_idx_acting']} took action {step_data['action_taken'].name}")
        print(f"Individual Rewards: {step_data['individual_rewards']}")
        print(f"Team Reward (Apples Picked): {step_data['team_reward']}")
        
        print("\nInitial State:")
        print("Apples:\n", np.array_str(step_data['initial_state'].apples))
        print("Agents:\n", np.array_str(step_data['initial_state'].agents))
        
        print("\nFinal State of Step:")
        print("Apples:\n", np.array_str(step_data['final_state_of_step'].apples))
        print("Agents:\n", np.array_str(step_data['final_state_of_step'].agents))
        print("-" * 30)
# print_evaluation_trace(nearest_data, num_steps_to_print=100)

In [ ]:
print("\n--- Checkpoints ---")
for i in range(NUM_ENV_CHECKPOINTS):
    apples = apples_in_env[i] if i < len(apples_in_env) else 0
    # log apples and states
    print(f"----------Checkpoint {i + 1}--------------")
    print(f"Apples: {apples}")
    print(f"States: {state_to_raw_dict(checkpoint_states[i]) if i < len(checkpoint_states) else {}}")

In [ ]:
if not Path(FILENAME).exists():
    with open(FILENAME, "w", newline="") as f:
        writer = csv.writer(f)
        # Add "conv_channels" and clarify "mlp_layers"
        writer.writerow(["run_name", "conv_channels", "mlp_layers", "width", "height", "mean_random_reward", "mean_nearest_reward", "mean_learned_reward"])

# --- AFTER ---
with open(FILENAME, "a", newline="") as f:
    writer = csv.writer(f)
    # Convert the list of channels to a hyphen-separated string
    conv_channels_str = '-'.join(map(str, CONV_CHANNELS))
    
    # Write the new string to the CSV
    writer.writerow(["decentralized", conv_channels_str, HIDDEN_LAYERS, WIDTH, HEIGHT, np.mean(random_rewards), np.mean(nearest_rewards), np.mean(learned_rewards)])